# CISB5123 Text Analytics - Lab Assignment 3
## Topic Modeling using LDA (Gensim)

**Name:** TUN DANIAL ADLI BIN TUN ALI  
**ID:** IS01083502

**Name:** Amirul Irfaan bin Mohd.Hishamuddin  
**ID:** IS01083864

In this assignment, we work with `news_dataset.csv` (unlabelled) and use LDA to discover **4 topics** from the `text` column.

## 1. Import the necessary libraries

In [2]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   -- ------------------------------------- 1.6/24.4 MB 7.7 MB/s eta 0:00:03
   ----- ---------------------------------- 3.4/24.4 MB 9.4 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 8.7 MB/s eta 0:00:03
   ---------- ----------------------------- 6.6/24.4 MB 8.2 MB/s eta 0:00:03
   ------------- -------------------------- 8.1/24.4 MB 8.4 MB/s eta 0:00:02
   --------------- ------------------------ 9.4/24.4 MB 7.8 MB/s eta 0:00:02
   ----------------- ---------------------- 10.5/24.4 MB 7.4 MB/s eta 0:00:02
   ------------------- -------------------- 12.1/24.4 MB 7.5 MB/s eta 0:00:02
   --------------------- ------------------ 13.1/24.4 MB 7.2 MB/s eta 0:00:02
   ----------------------- ---------------- 14.2/24.4 MB 7.0 MB/s eta 0:00:02
   ------------------------- -------------- 15.7/24.4 MB 7.1 MB/s eta 0:00:02
   --------------------------- ------------ 16.8/24.4 MB 6.9 MB/s eta 0:00:02


In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to C:\Users\Nyllx-
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Nyllx-
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Nyllx-
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Nyllx-
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Nyllx-
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 2. Read the data (use only the 'text' column)

In [5]:
df = pd.read_csv('news_dataset.csv')
df = df[['text']]
print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (11314, 1)


,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


## 3. Text Pre-processing
Steps performed:
- Remove null values
- Lowercase and remove non-alphabetic characters
- Tokenization
- Remove stopwords
- Stemming
- Lemmatization

In [6]:
# Remove null values
print("Null values before:", df['text'].isnull().sum())
df = df.dropna(subset=['text']).reset_index(drop=True)
print("Null values after:", df['text'].isnull().sum())
print("Shape after removing nulls:", df.shape)

Null values before: 218
Null values after: 0
Shape after removing nulls: (11096, 1)


In [7]:
# Initialize stopwords, stemmer, lemmatizer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Apply preprocessing
df['tokens'] = df['text'].apply(preprocess_text)
df[['text', 'tokens']].head()

,text,tokens
0,I was wondering if anyone out there could enli...,"[wonder, anyon, could, enlighten, car, saw, da..."
1,I recently posted an article asking what kind ...,"[recent, post, articl, ask, kind, rate, singl,..."
2,\nIt depends on your priorities. A lot of peo...,"[depend, prioriti, lot, peopl, put, higher, pr..."
3,an excellent automatic can be found in the sub...,"[excel, automat, found, subaru, legaci, switch..."
4,: Ford and his automobile. I need information...,"[ford, automobil, need, inform, whether, ford,..."


## 4. Perform LDA using Gensim

In [8]:
# Create the dictionary and corpus
texts = df['tokens'].tolist()
dictionary = corpora.Dictionary(texts)

# Filter out very rare and very common words
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create the Bag-of-Words corpus
corpus = [dictionary.doc2bow(text) for text in texts]

print("Number of unique tokens:", len(dictionary))
print("Number of documents:", len(corpus))

Number of unique tokens: 10989
Number of documents: 11096


In [9]:
# Build the LDA model with 4 topics
num_topics = 4

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    iterations=100,
    alpha='auto',
    eta='auto'
)

# Print the topics
print("Top words per topic:\n")
for idx, topic in lda_model.print_topics(num_words=10):
    print(f"Topic {idx}: {topic}\n")

Top words per topic:

Topic 0: 0.015*"key" + 0.015*"use" + 0.009*"encrypt" + 0.008*"system" + 0.007*"chip" + 0.007*"edu" + 0.006*"file" + 0.006*"secur" + 0.006*"inform" + 0.005*"mail"

Topic 1: 0.009*"peopl" + 0.009*"would" + 0.009*"one" + 0.006*"say" + 0.006*"think" + 0.006*"know" + 0.005*"like" + 0.005*"govern" + 0.005*"right" + 0.004*"time"

Topic 2: 0.025*"max" + 0.011*"use" + 0.009*"window" + 0.008*"one" + 0.008*"get" + 0.006*"problem" + 0.006*"like" + 0.006*"file" + 0.005*"time" + 0.005*"work"

Topic 3: 0.009*"year" + 0.008*"game" + 0.007*"team" + 0.006*"presid" + 0.006*"stephanopoulo" + 0.005*"play" + 0.005*"new" + 0.004*"last" + 0.004*"player" + 0.004*"first"



## 5. Evaluate the LDA model using Coherence score

In [10]:
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model_lda.get_coherence()
print(f"Coherence Score (c_v): {coherence_score:.4f}")

Coherence Score (c_v): 0.5022


## 6. Interpretation of the Result

The LDA model was trained with **4 topics** on the preprocessed `text` column of the news dataset. Each topic is represented by a group of words with their probability weights, where higher-probability words are more representative of the topic.

### Interpretation of the Coherence Score

The coherence score (using the `c_v` measure) evaluates how semantically similar the top words within each topic are to each other. The score ranges between 0 and 1, where higher values indicate more interpretable and meaningful topics. A coherence score around **0.3–0.4** is considered acceptable, **0.4–0.55** is good, and values above **0.55** indicate strong, well-separated topics. Our model achieved a coherence score shown above, which suggests that the 4 topics discovered by the LDA model are reasonably coherent the top words within each topic tend to co-occur meaningfully across documents, making the topics interpretable. A lower score would imply that the topic words are too generic or unrelated, while a very high score would indicate highly distinctive and meaningful groupings. The result confirms that LDA successfully grouped the news articles into four distinguishable thematic clusters based on word co-occurrence patterns.